# 종합 리포트 — 설계·검증 시각화

> **버전 추적형 User RAG (개인 맞춤형 피드백 루프)** · 담당: **경이** (LangGraph 회의 워크플로 · 점수 엔진 · 종합 리포트)
> 용도: 최종 보고서 · 평가 리포트 · 발표 PPT 삽입용 (각 차트는 PNG로 자동 저장됨)
>
> **원칙: 이 노트북의 모든 수치는 레포에서 실측된 값이다.** 지어낸 벤치마크 없음.
> 재현 경로는 `docs/devlogs/mky-devlog.md` (2026-07-23 ~ 07-26) 참고.

---

## 파이프라인 개요 — 공고문이 유일한 기준

```
① 공고문 업로드          (파일명 '공고문' = 중심 자료, 배점표 포함 원문)
        │
② 동적 rubric 추출 🛡️    사전 가드 → LLM(temp=0·seed) → 항목명·배점 원문 확인·강제 정렬
        │
③ LangGraph 위원 회의     위원별 독립 채점 · 판정 밴드 캘리브레이션
        │
④ 결정론 보정 🛡️          근거 신호 기반 점수 상한 · 지적 인용 원문 검증(issue_refs)
        │
⑤ 점수 엔진               측정 가능 항목만 합산 → 만점 = 배점 합 (예: 85점)
        │
⑥ 버전 히스토리           v1.0 → v1.n 누적 · 해결/신규/잔존 지적 추적
        │
⑦ 종합 리포트 UI          점수 체계표 · 우선순위 · 4단계 근거 플로우
```

🛡️ = **결정론적 검증 게이트**: LLM 출력을 코드가 공고문·제출 문서 **원문과 대조** — LLM이 지어내도 코드가 거부·보정한다.

In [ ]:
# 공통 설정 — 한글 폰트(Windows: Malgun Gothic) + 팔레트 + PNG 저장 헬퍼
import matplotlib.pyplot as plt
import matplotlib as mpl
from pathlib import Path

mpl.rcParams['font.family'] = 'Malgun Gothic'
mpl.rcParams['axes.unicode_minus'] = False
mpl.rcParams['figure.dpi'] = 110

PURPLE, GRAY, RED, GREEN, BEIGE = '#7c5cea', '#c9c4d4', '#e0603d', '#16a37a', '#faf8f4'
OUT = Path('2026_07_25_시각화자료')  # PNG 저장 폴더(팀 정리 구조) → PPT에 바로 삽입
OUT.mkdir(exist_ok=True)

def save(fig, name):
    fig.savefig(OUT / name, bbox_inches='tight', dpi=200, facecolor='white')
    print(f'저장: {OUT / name}')

## 실측 1 — 재현성: 같은 문서는 같은 점수

같은 문서·같은 rubric을 재채점했을 때의 총점. 개선 전에는 **±32.5** 흔들렸고(경계 문서에서 LLM 판정 flip),
`temperature=0` + `seed` 고정 + 결정론 점수 엔진 적용 후 **±1.5**로 안정.
재현성이 확보돼야 버전별 점수 추이(상승/하락)가 의미를 가진다.

In [ ]:
# 실측값: 개선 전 31.5 ↔ 64 / 개선 후 54.5, 56, 55.5 (같은 문서 재채점)
labels = ['개선 전\n1회차', '개선 전\n2회차', '개선 후\n1회차', '개선 후\n2회차', '개선 후\n3회차']
scores = [31.5, 64, 54.5, 56, 55.5]
colors = [GRAY, GRAY, PURPLE, PURPLE, PURPLE]

fig, ax = plt.subplots(figsize=(8.6, 4.4))
bars = ax.bar(labels, scores, color=colors, width=0.62)
ax.bar_label(bars, fmt='%.1f', fontsize=11, fontweight='bold', padding=3)
ax.set_ylim(0, 80)
ax.set_ylabel('총점')
ax.set_title('재현성 — 같은 문서 재채점 총점 (temperature=0 + seed 고정)', fontsize=13, fontweight='bold', pad=12)
ax.spines[['top', 'right']].set_visible(False)
ax.axhspan(31.5, 64, xmin=0.02, xmax=0.38, color=RED, alpha=0.08)
ax.annotate('편차 ±32.5', xy=(0.5, 66), ha='center', color=RED, fontsize=11, fontweight='bold')
ax.annotate('편차 ±1.5', xy=(3, 61), ha='center', color=PURPLE, fontsize=11, fontweight='bold')
save(fig, 'fig1_reproducibility.png')
plt.show()

## 실측 2 — 변별력: 품질 차이가 점수 차이로

같은 공고문·같은 rubric으로 **저품질/우수 문서 쌍**을 채점.
판정 밴드 캘리브레이션 + [엄정 채점] 강제 + 근거 신호 기반 결정론 점수 상한 적용.
핵심은 **역검증**: 채점을 엄격하게 만들어도 **우수 문서는 깎이지 않아야** 한다(데이터 항목 55% vs 90%).

In [ ]:
# 실측값: (1차) 근거 없는 문서 21.5 vs 잘 쓴 문서 77 / (2차 조건) 55.5 vs 81
pairs = ['1차 검증', '2차 검증(역검증 포함)']
bad, good = [21.5, 55.5], [77, 81]
x = range(len(pairs)); w = 0.32

fig, ax = plt.subplots(figsize=(8.2, 4.4))
b1 = ax.bar([i - w/2 for i in x], bad, w, color=RED, label='저품질 문서(출처·수치 전무)')
b2 = ax.bar([i + w/2 for i in x], good, w, color=GREEN, label='우수 문서(출처·수치·방법 구체)')
ax.bar_label(b1, fmt='%.1f', fontsize=11, fontweight='bold', padding=3)
ax.bar_label(b2, fmt='%.0f', fontsize=11, fontweight='bold', padding=3)
ax.set_xticks(list(x), pairs)
ax.set_ylim(0, 100)
ax.set_ylabel('총점')
ax.set_title('변별력 — 같은 rubric에서 문서 품질별 총점 (캘리브레이션·엄정 채점·상한)', fontsize=13, fontweight='bold', pad=12)
ax.legend(frameon=False, fontsize=10)
ax.spines[['top', 'right']].set_visible(False)
save(fig, 'fig2_discrimination.png')
plt.show()

## 실측 3 — 기준 충실성: 만점은 공고문 배점표가 결정 (NIA 실증·PoC)

실제 발생한 추출 사고: 부문별 배점 열이 2개(실증·PoC / 우수사례)인 표에서 LLM이 일부 항목만
다른 열을 채택(확산성 25, 안전윤리 15) → **결정론 보정**이 공고문 원문의 첫 부문 열 값으로 강제 정렬.
주관 항목(안전성·윤리성)은 채점에서 제외(사유 공개) → **만점 = 15+30+30+10 = 85점** (100점 아님).

In [ ]:
# 실측값: 공고문 배점(실증·PoC 열) vs LLM 추출 사고 vs 결정론 보정 후
items = ['목표\n부합성', '기술성·\n혁신성', '실현\n가능성', '확산성·\n효과성', '안전성·\n윤리성']
notice  = [15, 30, 30, 10, 10]   # 공고문 원문(실증·PoC 열)
wrong   = [15, 30, 30, 25, 15]   # 추출 사고(열 혼용)
fixed   = [15, 30, 30, 10, 10]   # 결정론 보정 후
x = range(len(items)); w = 0.26

fig, ax = plt.subplots(figsize=(9.4, 4.6))
b1 = ax.bar([i - w for i in x], notice, w, color=PURPLE, label='공고문 원문 배점')
b2 = ax.bar(list(x), wrong, w, color=GRAY, label='추출 사고(열 혼용)')
b3 = ax.bar([i + w for i in x], fixed, w, color=GREEN, label='결정론 보정 후')
for b in (b1, b2, b3): ax.bar_label(b, fontsize=9.5, padding=2)
ax.set_xticks(list(x), items)
ax.set_ylabel('배점')
ax.set_title('기준 충실성 — 배점을 공고문 원문 값으로 강제 정렬 (확산성 25→10 · 안전윤리 15→10)', fontsize=12.5, fontweight='bold', pad=12)
ax.legend(frameon=False, fontsize=10)
ax.spines[['top', 'right']].set_visible(False)
# 안전성·윤리성 = 채점 제외 표시
ax.annotate('채점 제외\n(주관 항목·사유 공개)', xy=(4, 17), ha='center', fontsize=9.5, color=RED, fontweight='bold')
save(fig, 'fig3_rubric_fidelity.png')
plt.show()

print(f'만점(측정 가능 항목 합) = 15+30+30+10 = {15+30+30+10}점  ← 100점이 아님(주관 항목·가점 제외)')

## 실측 4 — 근거 기반성: 모든 항목 채점은 실제 원문 인용에 연결

최신 회의(2026-07-26 실측 스냅샷)의 **실제 evidence 연결 현황**.
왼쪽: 채점된 4개 항목 모두 제출 문서·공고문(중심)·보조 자료의 **실제 원문 인용**에 연결돼 있다
(안전성·윤리성은 채점 제외라 없음 — 설계 의도대로).
오른쪽: 지적 속 인용 14건을 **제출 문서 원문과 대조** — 검증 통과 6건만 화면에 표시되고,
원문과 일치하지 않는 8건은 **게이트가 차단**했다. 지어낸·부정확 인용이 사용자에게 노출되지 않음을
실데이터로 실증하는 수치다(차단이 0건이면 게이트가 필요 없다는 뜻이고, 차단이 있다는 것이 게이트의 존재 이유).

In [ ]:
# 실측 스냅샷(2026-07-26, 최신 회의): 항목별 근거 인용 수(출처 유형별) + issue_refs 원문 검증 결과
# 산출 방법: meetings.rubric_scores[].evidence_ids → evidence(quote·문서명) 조인, 문서 역할별 집계.
#            issue_refs는 지적 속 인용을 제출 문서 원문과 대조한 verified 값 집계. (devlog 재현 경로)
import numpy as np
crits = ['목표\n부합성', '기술성·\n혁신성', '실현\n가능성', '확산성·\n효과성']
submission = [2, 1, 2, 2]   # 제출 문서 인용 수
notice_c   = [0, 1, 0, 2]   # 공고문(중심 자료) 인용 수
support    = [2, 2, 2, 0]   # 보조 자료 인용 수

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11.5, 4.4), width_ratios=[1.5, 1])
x = np.arange(len(crits))
ax1.bar(x, submission, 0.55, color=PURPLE, label='제출 문서')
ax1.bar(x, notice_c, 0.55, bottom=submission, color=GREEN, label='공고문(중심 자료)')
ax1.bar(x, support, 0.55, bottom=np.array(submission) + np.array(notice_c), color=GRAY, label='보조 자료')
for i in x:
    total = submission[i] + notice_c[i] + support[i]
    ax1.text(i, total + 0.12, str(total), ha='center', fontsize=11, fontweight='bold')
ax1.set_xticks(x, crits)
ax1.set_ylabel('연결된 근거 인용 수')
ax1.set_ylim(0, 6)
ax1.set_title('항목별 채점 근거 연결 현황 (실측 스냅샷)', fontsize=12.5, fontweight='bold', pad=10)
ax1.legend(frameon=False, fontsize=9.5, loc='upper right')
ax1.spines[['top', 'right']].set_visible(False)

# 오른쪽: issue_refs 원문 검증 게이트 — 통과 6 / 차단 8 (실측, 총 14건)
gate_labels = ['검증 통과\n→ 화면 표시', '원문 불일치\n→ 표시 차단']
gate_vals = [6, 8]
bars = ax2.bar(gate_labels, gate_vals, 0.5, color=[GREEN, RED])
ax2.bar_label(bars, fontsize=12, fontweight='bold', padding=3)
ax2.set_ylim(0, 10)
ax2.set_ylabel('지적 인용 수')
ax2.set_title('지적 인용 원문 검증 게이트 (총 14건)', fontsize=12.5, fontweight='bold', pad=10)
ax2.spines[['top', 'right']].set_visible(False)
fig.suptitle('근거 기반성 — 채점은 실제 원문에만 연결되고, 검증 실패 인용은 차단된다', fontsize=13.5, fontweight='bold', y=1.04)
save(fig, 'fig4_groundedness.png')
plt.show()

## 실측 5 — 할루시네이션 3중 방어: 실제 사고에서 얻은 게이트

| # | 실제 사고 | 차단 장치(결정론) | 결과 |
|---|---|---|---|
| ① | 잘못된 URL 스크랩(43자 잡문)인데 "기본 4범주×25점" rubric이 '추출 성공' 표시 | **사전 가드** — 평가·배점 신호 없으면 LLM 호출 자체 차단 | 폴백 + "공고문 기준 아님" 경고 |
| ② | 신청서식의 작성 항목 제목 4개가 25점 균등 배분으로 rubric화 | **배점 숫자 원문 확인** — 과반 미확인 시 추출 거부 | 거부 (실측 unmatched 4/4) |
| ③ | 부문 열 혼용(확산성 25 ← 우수사례 열) | **원문 첫 부문 열 값으로 강제 정렬** | 25→10 보정, 만점 85 재현 |

### 지적↔원문 1:1 검증 (issue_refs)

| 입력(지적 속 인용) | 판별 | 화면 표시 |
|---|---|---|
| 원문에 실제 있는 인용 | `verified: true` | STEP 1 "제출 문서 근거"로 표시 |
| 원문에 없는(지어낸) 인용 | `verified: false` | **표시되지 않음** |
| 인용 없는 지적 | ref 없음 | 항목 단위 인용으로 폴백 |

모든 인용은 색인된 실제 문서 원문(RAG evidence)에서만 렌더되고, **KURE 의미 유사도순** 상위만 표시 — 프론트가 문장을 생성하지 않는다.

## 실측 6 — 판정 표시 일관성: 판정은 최종 점수 비율에서 결정론 도출 (2026-07-27 추가)

사용자 테스트에서 발견된 모순: **판정 「적정」인데 점수는 15점 중 6점(40%)**. 위원 LLM의 원 판정이
근거 상한(calibration) 적용 **전** 기준이라 최종 점수와 어긋났고, 재분석마다 판정이 적정↔보완 필요로
흔들려 접속 시점마다 달라 보였다.

화면 판정을 **최종(보정 후) 점수 비율**에서 결정론 도출하도록 변경 — 85%↑ 우수 / 65%↑ 적정 /
40%↑ 보완 필요 / 미만 중대 리스크 (reviewer_prompt 판정 밴드와 동일 경계). 같은 점수 = 언제나 같은
판정이고, 우선순위 정렬·팝업·STEP 3·점수 변화 팝업이 한 곳에서 계산되어 일괄 일치한다.

> 2026-07-27 추가분 PNG는 `2026_07_27_시각화자료/`에 저장된다 (날짜별 정리 구조).

In [ ]:
# 2026-07-27 추가분 저장 폴더(날짜별 정리 구조)
OUT2 = Path('2026_07_27_시각화자료')
OUT2.mkdir(exist_ok=True)

def save2(fig, name):
    fig.savefig(OUT2 / name, bbox_inches='tight', dpi=200, facecolor='white')
    print(f'저장: {OUT2 / name}')

INK, OCHRE = '#1c1a2e', '#c98a1b'

# 실측 사례: 목표 부합성 6/15점 = 40% — 개선 전 「적정」 표시(모순), 개선 후 「보완 필요」 고정
fig, ax = plt.subplots(figsize=(10.2, 3.9))
bands = [
    (0, 40, RED, '중대 리스크\n0~40%'),
    (40, 65, OCHRE, '보완 필요\n40~65%'),
    (65, 85, PURPLE, '적정\n65~85%'),
    (85, 100, GREEN, '우수\n85~100%'),
]
for x0, x1, c, label in bands:
    ax.barh(0, x1 - x0, left=x0, height=0.5, color=c, alpha=0.30, edgecolor='white', linewidth=2)
    ax.text((x0 + x1) / 2, 0, label, ha='center', va='center', fontsize=10.5, fontweight='bold', color=INK)
ax.axvline(40, color=INK, lw=1.8, ls='--', ymin=0.28, ymax=0.72)
ax.annotate('실측 사례 — 목표 부합성 6/15점 = 40%\n개선 후: 최종(보정 후) 점수 비율로 결정론 판정 → 항상 「보완 필요」 (경계 40%는 보완 필요에 포함)',
            xy=(40, 0.27), xytext=(44, 0.66), fontsize=10, fontweight='bold', color=INK,
            arrowprops=dict(arrowstyle='->', color=INK, lw=1.4))
ax.text(1, -0.62, '개선 전: 같은 6/15점인데 「적정」 표시 — LLM 원 판정이 근거 상한(calibration) 적용 전 기준이라 모순\n           + 재분석마다 적정↔보완 필요로 흔들려 접속 시점마다 판정이 달라 보임',
        fontsize=9.8, color=RED, fontweight='bold', va='top')
ax.set_xlim(0, 100)
ax.set_ylim(-1.05, 1.05)
ax.set_yticks([])
ax.set_xticks([0, 40, 65, 85, 100])
ax.set_xlabel('최종(보정 후) 점수 / 배점 (%)')
ax.set_title('판정 표시 일관성 — 판정은 최종 점수 비율에서 결정론 도출 (reviewer_prompt 밴드와 동일 경계)',
             fontsize=12.5, fontweight='bold', pad=12)
ax.spines[['top', 'right', 'left']].set_visible(False)
save2(fig, 'fig5_judgment_consistency.png')
plt.show()

## 실측 7 — 버전 추적 정확성: '해결됨'은 직전 버전 지적에 근거할 때만 (2026-07-26~27 추가)

재분석마다 LLM이 같은 지적을 **다른 문장으로 재표현**하면, 집합 비교(문자열 일치)가 "이전 것 해결됨 +
사실상 같은 지적 신규"로 잘못 갈라 **모순 동시 표시**되던 문제(사용자 테스트에서 발견).

- **해결됨 근거 검증**: resolved_issues가 직전 버전의 실제 지적과 매칭될 때만 표시(2-gram Dice ≥ 0.66
  또는 포함), 직전 지적 1건당 1건으로 합침 — **표시 6건 → 2건 (v1.0→v1.1 실측)**. 문구는 재표현본이
  아니라 **직전 버전 지적 원문**을 취소선으로("v1.0 보완 필요 …" + 끝에 해결됨). 직전 버전에 없던
  해결됨은 근거 없음 → 미표시.
- **재표현 잔존 매칭**: 지적문에서 일반어를 뺀 **핵심 주제 토큰**(법·제도/예산/모델·알고리즘…)이 절반
  이상 겹치면 잔존(보완 필요)으로 표시하고 가짜 해결됨/신규 제거. 오합병 검증: "확산 계획"↔"성과
  지표(KPI)"는 주제 겹침 0 → 구분 유지.

In [ ]:
# 실측값(v1.0→v1.1): 표시된 '해결됨' 6건 → 직전 지적 매칭 검증 통과 2건
# 재표현 판별(2-gram Dice, 실측): 같은 지적 어미 변형 0.82·0.64(포함 규칙 매칭) / 다른 지적 0.55 — 임계 0.66
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11.8, 4.5), width_ratios=[1, 1.45])

bars = ax1.bar(['개선 전\n표시된 해결됨', '개선 후\n근거 검증 통과'], [6, 2], 0.5, color=[GRAY, GREEN])
ax1.bar_label(bars, fontsize=13, fontweight='bold', padding=3)
ax1.set_ylim(0, 8)
ax1.set_ylabel("'해결됨' 표시 건수")
ax1.set_title("해결됨 근거 검증 (v1.0→v1.1 실측)", fontsize=12, fontweight='bold', pad=10)
ax1.text(0.5, -0.24, '직전 버전 지적과 매칭될 때만 · 1건당 1건 합침\n직전 버전에 없던 해결됨은 근거 없음 → 미표시',
         transform=ax1.transAxes, ha='center', va='top', fontsize=9.5, color=INK)
ax1.spines[['top', 'right']].set_visible(False)

names = ["같은 지적\n어미 변형 ①", "같은 지적\n어미 변형 ②", "다른 지적\n확산 계획 ↔\n성과 지표(KPI)"]
vals = [0.82, 0.64, 0.55]
b = ax2.barh(names, vals, 0.52, color=[GREEN, GREEN, RED])
ax2.bar_label(b, fmt='%.2f', fontsize=11, fontweight='bold', label_type='center', color='white')
ax2.axvline(0.66, color=INK, lw=1.8, ls='--')
ax2.set_ylim(2.85, -0.5)  # 축 반전 + 아래쪽에 임계 라벨 여백
ax2.text(0.66, 2.55, '매칭 임계 0.66', ha='center', va='center', fontsize=10, fontweight='bold', color=INK)
ax2.annotate('임계 미만이지만 한쪽이 다른 쪽을\n포함(포함 규칙) → 같은 지적으로 매칭',
             xy=(0.655, 1), xytext=(0.74, 1.72), fontsize=9.3, color=GREEN, fontweight='bold',
             arrowprops=dict(arrowstyle='->', color=GREEN, lw=1.2))
ax2.set_xlim(0, 1.0)
ax2.tick_params(axis='y', labelsize=10)
ax2.set_xlabel('지적문 2-gram Dice 유사도 (실측)')
ax2.set_title('재표현 판별 캘리브레이션 — 같은 지적은 흡수, 다른 지적은 구분', fontsize=12, fontweight='bold', pad=10)
ax2.spines[['top', 'right']].set_visible(False)
fig.subplots_adjust(wspace=0.45)

fig.suptitle("버전 추적 정확성 — '해결됨'은 직전 버전 지적에 근거할 때만 표시된다", fontsize=13.5, fontweight='bold', y=1.04)
save2(fig, 'fig6_version_tracking.png')
plt.show()

## 실측 8 — 2차 근거 게이트 · 정직 폴백 (2026-07-26~27 추가)

전부 **표시 계층 결정론**(저장 데이터 불변 · 기존 회의에도 소급 적용). 각 행은 실제 발생한 사고/문제의 실측 기록이다.

| 실제 사고/문제 (실측) | 차단 장치 | 결과 |
|---|---|---|
| 붙임2 p.9에 없는 문장(rubric 설명문)이 실제 파일명·페이지에 붙어 인용 표시 | **인용 원문 게이트** — evidence의 quote(위원 LLM 자기보고)가 색인된 청크 원문에 실제 존재할 때만 표시, 아니면 청크 원문으로 대체·청크 미상 제외 | 지어낸 인용이 화면에 나오지 않음 |
| '법적 제약' 지적 밑에 '구현 서비스 혁신성' 문단이 인용됨 | **STEP 1 지적-근거 정합** — 지적 주제 토큰이 있는 인용만 표시(지적 전용 검증 인용은 항상), 없으면 "내용 부재" 정직 폴백 | 지적과 무관한 인용 미표시 |
| 'AI 모델·알고리즘 설계 적정성' 섹션이 실현 가능성 항목에도 중복 표시 | **보조 자료 배타 배정** — 섹션이 의미상 가장 맞는 평가 항목에서만(제외 항목도 배정 후보) | 기술성·혁신성에만 표시 |
| 옛 버전 클릭 시 최신 문서의 AI 피드백(오탈자·분량)이 그대로 표시(오귀속) | **버전별 스냅샷** — 검사 결과를 그 문서를 분석한 회의(=버전)에 저장, 스냅샷 없는 옛 버전은 "기록 없음" 정직 안내 | 버전별 정확 귀속 |
| 분량 기준 "기준 없음" — 실제로는 붙임2 서식에 "최대 30p 이내" 존재 | **전체 공고 자료 합본 추출** — 첫 공고 문서만이 아니라 모든 criteria 문서 합본에서 추출 | "20p / 30p" 정상 표기 (실측) |

## 회귀 안전망 · 발표 핵심 메시지

- **자동 테스트**: `ai/meeting/tests/` **366개 전부 통과** 유지 (동적 rubric 검증 · 배점 보정 · 버전 비교 · calibration 포함)
- **계약 안정성**: `review_output.schema.json` 은 선택 필드 추가만(v1.1 규칙) — `extracted_from_notice` · `excluded_criteria` · `issue_refs`
- **역할 구분**: 검색 품질(Context Precision/Recall, Ragas 배치 측정)은 RAG 파이프라인(용준) 영역 — 종합 리포트는 그 출력을 근거로 소비

> **핵심 메시지**: "종합 리포트는 **공고문 원문만이 기준**이 되도록 3중 결정론 검증으로 지어내기를 차단하고,
> **seed 고정 + 결정론 점수 엔진**으로 같은 문서엔 같은 점수를, **캘리브레이션**으로 품질 차이엔 점수 차이를 보장한다.
> 모든 지적은 **제출 문서의 실제 문장(p.N)** 을 근거로 제시되며, 검증에 실패한 근거는 화면에 나오지 않는다."